In [ ]:
import random
from datasets import load_dataset, get_dataset_config_names
import os
from typing import Dict, Any

# 🌟 데이터셋 메타 정보 요약 및 목표 설정 🌟
# 데이터셋명: nayohan/Magpie-Pro-MT-300K-v0.1-ko
# 의미: 한국어(ko)로 생성된 대규모 번역/대화 학습 데이터셋입니다.
# 목표: 이 데이터는 '입력(Prompt)'과 '출력(Response)'의 쌍으로 구성되어 있어,
#       LLM에게 좋은 답변을 하도록 가르치는 '지침(Instruction)' 튜닝 데이터가 핵심입니다.
#       우리의 실습 목표는 이 대화 형식(conversations)의 구조를 파악하고,
#       가장 효과적인 '질문-답변 프롬프트'를 자동으로 생성하는 원리를 탐구하는 것입니다.

# --- 설정 상수 ---
DATASET_ID = "nayohan/Magpie-Pro-MT-300K-v0.1-ko"
SPLIT_NAME = "train"
SAMPLE_COUNT = 10  # 실습 시 출력할 샘플의 개수 (데이터셋 전체를 다룰 필요는 없어요!)
ANALYSIS_COUNT = 100 # 통계 분석을 위해 로드할 샘플의 개수 (더 많은 데이터를 활용해 봅시다!)

print("✨ 안녕! 파이썬 AI 튜터가 왔어! 🚀")
print("오늘의 미션은 거대한 대화 데이터셋 속에서 보석 같은 프롬프트를 찾아내는 거야. 걱정 마, 너만 할 수 있어!")
print("-" * 50)

# --- 1. 데이터 로드 준비 ---
dataset = None

try:
    # 💡 꿀팁: 스트리밍 모드(streaming=True)를 사용하면,
    # 수십 GB의 대용량 데이터셋을 다운로드하지 않고 '미리보기'만 할 수 있어. 빨라!
    print(f"🔍 1. 스트리밍 모드로 '{DATASET_ID}' 데이터를 로드 시도 중...")
    dataset = load_dataset(DATASET_ID, split=SPLIT_NAME, streaming=True)
    print("✅ 스트리밍 모드 로드 성공! 👍")
    
except Exception as e:
    # ⚠️ 스트리밍이 실패하면(드물지만), 소량만 로드하는 방식으로 전환할게.
    print(f"❌ 스트리밍 모드 로드 중 오류 발생: {e}")
    print("🌐 일반 모드(streaming=False)로 전환하여 소량만 다운로드합니다...")
    try:
        dataset = load_dataset(DATASET_ID, split=SPLIT_NAME, streaming=False)
        print("✅ 일반 모드 로드 성공! 🎉")
    except Exception as e_fallback:
        print(f"🚨 모든 로드 시도 실패! 데이터셋 ID를 확인해 주세요. ({e_fallback})")
        exit()

# --- 2. 데이터 샘플링 전략 (스트리밍/일반 모드 구분) ---

# 📚 제약 조건에 따라, 데이터 전체를 순회하지 않고, 샘플만 추출해서 사용할 거야.
print("-" * 50)
print(f"🧩 2. {SAMPLE_COUNT}개의 샘플을 추출하여 분석을 준비합니다.")

# 🔗 제약 조건에 따라, 데이터셋이 스트리밍 객체인지 먼저 확인해야 해.
if hasattr(dataset, "take"):
    # .take()가 존재하면 스트리밍 데이터셋 (IterableDataset)
    print("🔍 스트리밍 데이터셋이 감지되었습니다. .take()를 사용합니다.")
    # iter(dataset.take(N)) 패턴을 사용해야 해!
    # Iterator를 만들고, 리스트로 변환해서 사용해야 분석하기 편해.
    sampled_dataset_iterator = iter(dataset.take(ANALYSIS_COUNT))
    
    # 리스트로 변환하여 분석할 데이터 묶음 만들기
    sample_data_list = list(sampled_dataset_iterator)
else:
    # 일반 데이터셋 (Dataset)
    print("🌐 일반 Dataset 객체로 감지되었습니다. .select()를 사용할 수 없습니다.")
    sample_data_list = list(dataset.select(range(ANALYSIS_COUNT)))

print(f"✅ 총 {len(sample_data_list)}개의 샘플을 성공적으로 준비했습니다.")

# --- 3. 주요 분석 실습: 대화 구조 파악 및 Prompt 엔지니어링 ---

def analyze_conversation_structure(sample_list: list) -> None:
    """
    데이터셋의 대화 구조를 분석하고, 핵심적인 패턴을 발견합니다.
    """
    print("\n" + "=" * 60)
    print("🧠 3. 대화 구조 분석: 어떤 역할(Role)이 가장 많이 쓰일까? (Diversity Check)")
    print("=" * 60)

    from_roles = []
    for sample in sample_list:
        # 'conversations' 필드는 리스트 형태의 딕셔너리들로 구성되어 있어.
        if 'conversations' in sample and sample['conversations']:
            # 첫 번째 대화 턴의 화자(from)를 기록해볼게.
            first_turn_role = sample['conversations'][0].get('from')
            if first_turn_role:
                from_roles.append(first_turn_role)

    unique_roles = set(from_roles)
    role_counts = {}
    for role in unique_roles:
        role_counts[role] = from_roles.count(role)

    print(f"🔎 데이터셋 전체의 대화 역할(Role) 종류: {len(unique_roles)}가지")
    print("📊 가장 자주 사용되는 대화 역할 상위 3가지:")
    
    # 통계를 위해 임시로 딕셔너리 리스트를 만들고 정렬
    sorted_roles = sorted(role_counts.items(), key=lambda item: item[1], reverse=True)
    
    for role, count in sorted_roles[:3]:
        print(f"  - 역할 '{role}': 등장 횟수 {count}회 (가장 많이 쓰이는 프롬프트 시작점일 수 있어요!)")

    print("\n✨ 분석 결론: 이 데이터셋은 '사용자(user)', 'AI 모델(ai)', '시스템(system)' 같은 명확한 역할을 학습하는 데 최적화되어 있어!")


def generate_example_prompts(sample_list: list, count: int) -> None:
    """
    선택된 샘플을 이용하여 사용자 친화적인 프롬프트 예시를 만듭니다.
    """
    print("\n" + "=" * 60)
    print(f"🎭 4. 초보자를 위한 실습: '{count}'개의 '꿀 프롬프트 예시' 만들기")
    print("=" * 60)
    
    # 랜덤 샘플을 뽑아보기 (너무 많은 샘플은 출력이 길어져서 어려우니 최소한만!)
    selected_samples = random.sample(sample_list, min(count, len(sample_list)))
    
    print("🚀 샘플을 로드할 때마다 모델이 어떤 대화를 했는지 확인해 볼 수 있어!")

    for i, sample in enumerate(selected_samples):
        if 'conversations' not in sample or not sample['conversations']:
            print(f"\n--- Sample {i+1}: (대화 기록 없음) ---")
            continue

        print(f"\n⭐ Sample {i+1}: 대화 시작 ⭐")
        
        # 대화 턴을 순회하면서 형식에 맞춰 출력
        for turn in sample['conversations']:
            from_role = turn.get('from', 'Unknown')
            value = turn.get('value', '').strip()
            
            # 🤓 친절한 주석: 여기서 우리가 할 일은 이 'value'를 가지고 모델에게 적절한 '프롬프트'를 만드는 거야.
            # 만약 'from_role'이 'user'라면, 이건 모델에게 던져야 할 '질문'이 되는 거지!
            
            print(f"[{'🧑‍💻 사용자' if from_role == 'user' else '🤖 모델' if from_role == 'ai' else from_role}]: {value[:80]}...") # 80자로 잘라서 출력
        
        print("--- ✨ 프롬프트 예시 끝. 다음 샘플로 넘어가 볼까? ✨ ---")


# --- 4. 실행 ---
if __name__ == "__main__":
    # 1. 대화 구조 분석 실행 (통계적 이해)
    analyze_conversation_structure(sample_data_list)

    # 2. 프롬프트 예시 생성 실행 (시각적/직관적 이해)
    generate_example_prompts(sample_data_list, SAMPLE_COUNT)
    
    print("\n" + "=" * 60)
    print("✨ 튜터 코멘트: 오늘 미션 완료! 🎊")
    print("이제 데이터셋의 구조를 이해했지? 이 데이터를 사용해서 너만의 멋진 챗봇을 만들 수 있을 거야. 화이팅! 💪")